In [0]:
# Import
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.sql.functions import (
    col,
    date_format,
    greatest,
    isnan,
    when,
    count,
    explode,
    arrays_zip,
    to_date,
    from_unixtime
)


In [0]:
scope_name = "traffic-kv-scope"

storage_account = "trafficcollision1storage"
container = "traffic-collision"

acc_storage_key = dbutils.secrets.get(
    scope=scope_name,
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    acc_storage_key
)

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

traffic_path = f"{base_path}/raw/toronto_traffic_collision/*.json"
weather_path = f"{base_path}/raw/weather/weather_raw.json"

## Read data

In [0]:
# Read Toronto weather JSON file from Open-Meteo
# Current raw path: traffic-collision/raw/weather/weather_raw.json

weather_path = f"{base_path}/raw/weather/weather_raw.json"

weather_df = spark.read.option("multiline", "true").json(weather_path)

# Flatten Open-Meteo daily arrays into one row per date
toronto_weather_raw = (
    weather_df
    .select(
        explode(
            arrays_zip(
                col("daily.time"),
                col("daily.precipitation_sum"),
                col("daily.temperature_2m_max")
            )
        ).alias("daily_record")
    )
    .select(
        col("daily_record.time").alias("time"),
        col("daily_record.precipitation_sum").alias("precipitation_sum"),
        col("daily_record.temperature_2m_max").alias("temperature_2m_max")
    )
)

toronto_weather_raw.show()


+----------+-----------------+------------------+
|      time|precipitation_sum|temperature_2m_max|
+----------+-----------------+------------------+
|2014-01-01|              0.0|              -9.2|
|2014-01-02|              2.5|             -15.2|
|2014-01-03|              0.0|             -12.4|
|2014-01-04|              0.1|              -1.7|
|2014-01-05|             11.0|               0.7|
|2014-01-06|              7.6|               3.3|
|2014-01-07|              0.2|             -13.8|
|2014-01-08|              0.0|              -8.5|
|2014-01-09|              0.0|              -7.2|
|2014-01-10|              1.4|               1.5|
|2014-01-11|              9.6|               5.5|
|2014-01-12|              0.0|               1.9|
|2014-01-13|              1.9|               5.4|
|2014-01-14|              0.2|               4.7|
|2014-01-15|              0.2|               0.7|
|2014-01-16|              0.8|              -1.4|
|2014-01-17|              0.4|               1.5|


In [0]:
toronto_weather_raw.printSchema()

root
 |-- time: string (nullable = true)
 |-- precipitation_sum: double (nullable = true)
 |-- temperature_2m_max: double (nullable = true)



In [0]:
toronto_weather_raw.summary().show()

+-------+----------+-----------------+------------------+
|summary|      time|precipitation_sum|temperature_2m_max|
+-------+----------+-----------------+------------------+
|  count|      4496|             4496|              4496|
|   mean|      NULL|2.384741992882576|12.522909252668981|
| stddev|      NULL|4.947875878242976| 10.60998271034871|
|    min|2014-01-01|              0.0|             -17.8|
|    25%|      NULL|              0.0|               3.7|
|    50%|      NULL|              0.2|              12.5|
|    75%|      NULL|              2.2|              22.0|
|    max|2026-04-23|             48.0|              35.8|
+-------+----------+-----------------+------------------+



In [0]:
# Read Toronto traffic collisions JSON files
# Current raw path contains one paginated JSON file per API offset.

traffic_path = f"{base_path}/raw/toronto_traffic_collision/*.json"

df_temp = spark.read.option("multiline", "true").json(traffic_path)

# Flatten ArcGIS JSON: features[*].attributes
toronto_collisions_raw = (
    df_temp
    .select(explode(col("features")).alias("feature"))
    .select("feature.attributes.*")
)

toronto_collisions_raw.show()


+----------+-------+--------+---------------+----------+--------------+--------+-----------------+------------------+------------------+----------+--------------------+--------+-------------+---------+--------+---------+--------+---------+-------------+----------+
|AUTOMOBILE|BICYCLE|DIVISION|EVENT_UNIQUE_ID|FATALITIES|FTR_COLLISIONS|HOOD_158|INJURY_COLLISIONS|         LAT_WGS84|        LONG_WGS84|MOTORCYCLE|   NEIGHBOURHOOD_158|OBJECTID|     OCC_DATE|  OCC_DOW|OCC_HOUR|OCC_MONTH|OCC_YEAR|PASSENGER|PD_COLLISIONS|PEDESTRIAN|
+----------+-------+--------+---------------+----------+--------------+--------+-----------------+------------------+------------------+----------+--------------------+--------+-------------+---------+--------+---------+--------+---------+-------------+----------+
|       YES|     NO|     D53| GO-20168037252|         0|            NO|     098|               NO|43.685876842418836|-79.39317876160105|        NO|Rosedale-Moore Pa...|  176001|1472616000000|Wednesday|    

In [0]:
toronto_collisions_raw.printSchema()

root
 |-- AUTOMOBILE: string (nullable = true)
 |-- BICYCLE: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- EVENT_UNIQUE_ID: string (nullable = true)
 |-- FATALITIES: long (nullable = true)
 |-- FTR_COLLISIONS: string (nullable = true)
 |-- HOOD_158: string (nullable = true)
 |-- INJURY_COLLISIONS: string (nullable = true)
 |-- LAT_WGS84: double (nullable = true)
 |-- LONG_WGS84: double (nullable = true)
 |-- MOTORCYCLE: string (nullable = true)
 |-- NEIGHBOURHOOD_158: string (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- OCC_DATE: long (nullable = true)
 |-- OCC_DOW: string (nullable = true)
 |-- OCC_HOUR: string (nullable = true)
 |-- OCC_MONTH: string (nullable = true)
 |-- OCC_YEAR: string (nullable = true)
 |-- PASSENGER: string (nullable = true)
 |-- PD_COLLISIONS: string (nullable = true)
 |-- PEDESTRIAN: string (nullable = true)



In [0]:
toronto_collisions_raw.summary().show()

+-------+----------+-------+--------+---------------+--------------------+--------------+------------------+-----------------+-----------------+------------------+----------+--------------------+------------------+--------------------+---------+------------------+---------+------------------+---------+-------------+----------+
|summary|AUTOMOBILE|BICYCLE|DIVISION|EVENT_UNIQUE_ID|          FATALITIES|FTR_COLLISIONS|          HOOD_158|INJURY_COLLISIONS|        LAT_WGS84|        LONG_WGS84|MOTORCYCLE|   NEIGHBOURHOOD_158|          OBJECTID|            OCC_DATE|  OCC_DOW|          OCC_HOUR|OCC_MONTH|          OCC_YEAR|PASSENGER|PD_COLLISIONS|PEDESTRIAN|
+-------+----------+-------+--------+---------------+--------------------+--------------+------------------+-----------------+-----------------+------------------+----------+--------------------+------------------+--------------------+---------+------------------+---------+------------------+---------+-------------+----------+
|  count|    

## Cleaning Toronto weather data

In [0]:
# Select only the weather fields available from the Open-Meteo raw JSON
# Your current API request includes precipitation_sum and temperature_2m_max.

toronto_weather_transformed = toronto_weather_raw


In [0]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import StringType

null_checks = []

for field in toronto_weather_transformed.schema.fields:
    c = field.name
    
    if isinstance(field.dataType, StringType):
        null_checks.append(
            count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
        )
    else:
        null_checks.append(
            count(when(col(c).isNull(), c)).alias(c)
        )

toronto_weather_transformed.select(null_checks).show()

+----+-----------------+------------------+
|time|precipitation_sum|temperature_2m_max|
+----+-----------------+------------------+
|   0|                0|                 0|
+----+-----------------+------------------+



In [0]:
# Remove null records

toronto_weather_transformed = toronto_weather_transformed.dropna(how="any")


In [0]:
# Remove any duplicate weather records for a specific day, if they exist

toronto_weather_transformed = toronto_weather_transformed.dropDuplicates(
    subset=["time"]
)


In [0]:
# Cast and rename weather columns for transformed layer / SQL-friendly schema

toronto_weather_transformed = (
    toronto_weather_transformed
    .withColumn("time", to_date(col("time")))
    .withColumnRenamed("time", "date")
    .withColumn("precipitation_sum", col("precipitation_sum").cast("float"))
    .withColumnRenamed("precipitation_sum", "precipitation_sum_mm")
    .withColumn("temperature_2m_max", col("temperature_2m_max").cast("float"))
    .withColumnRenamed("temperature_2m_max", "temperature_2m_max_c")
)


In [0]:
toronto_weather_transformed.printSchema()

root
 |-- date: date (nullable = true)
 |-- precipitation_sum_mm: float (nullable = true)
 |-- temperature_2m_max_c: float (nullable = true)



In [0]:
toronto_weather_transformed = toronto_weather_transformed.toDF(
    *[c.upper() for c in toronto_weather_transformed.columns]
)

In [0]:
toronto_weather_transformed = toronto_weather_transformed.sort("DATE", ascending=False)


In [0]:
toronto_weather_transformed.show()

+----------+--------------------+--------------------+
|      DATE|PRECIPITATION_SUM_MM|TEMPERATURE_2M_MAX_C|
+----------+--------------------+--------------------+
|2026-04-23|                 0.0|                14.8|
|2026-04-22|                 0.0|                15.5|
|2026-04-21|                 0.4|                 9.7|
|2026-04-20|                 0.0|                 4.1|
|2026-04-19|                 1.6|                 7.4|
|2026-04-18|                 5.4|                19.1|
|2026-04-17|                 0.0|                16.3|
|2026-04-16|                 6.2|                19.3|
|2026-04-15|                 4.0|                18.0|
|2026-04-14|                 3.7|                21.4|
|2026-04-13|                 2.5|                21.0|
|2026-04-12|                 2.2|                 8.8|
|2026-04-11|                 0.0|                 8.9|
|2026-04-10|                 5.8|                 7.6|
|2026-04-09|                 2.4|                15.4|
|2026-04-0

## Cleaning toronto traffic collisions data

In [0]:
toronto_collisions_raw.printSchema()

root
 |-- AUTOMOBILE: string (nullable = true)
 |-- BICYCLE: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- EVENT_UNIQUE_ID: string (nullable = true)
 |-- FATALITIES: long (nullable = true)
 |-- FTR_COLLISIONS: string (nullable = true)
 |-- HOOD_158: string (nullable = true)
 |-- INJURY_COLLISIONS: string (nullable = true)
 |-- LAT_WGS84: double (nullable = true)
 |-- LONG_WGS84: double (nullable = true)
 |-- MOTORCYCLE: string (nullable = true)
 |-- NEIGHBOURHOOD_158: string (nullable = true)
 |-- OBJECTID: long (nullable = true)
 |-- OCC_DATE: long (nullable = true)
 |-- OCC_DOW: string (nullable = true)
 |-- OCC_HOUR: string (nullable = true)
 |-- OCC_MONTH: string (nullable = true)
 |-- OCC_YEAR: string (nullable = true)
 |-- PASSENGER: string (nullable = true)
 |-- PD_COLLISIONS: string (nullable = true)
 |-- PEDESTRIAN: string (nullable = true)



In [0]:
toronto_collisions_raw.show()

+----------+-------+--------+---------------+----------+--------------+--------+-----------------+------------------+------------------+----------+--------------------+--------+-------------+---------+--------+---------+--------+---------+-------------+----------+
|AUTOMOBILE|BICYCLE|DIVISION|EVENT_UNIQUE_ID|FATALITIES|FTR_COLLISIONS|HOOD_158|INJURY_COLLISIONS|         LAT_WGS84|        LONG_WGS84|MOTORCYCLE|   NEIGHBOURHOOD_158|OBJECTID|     OCC_DATE|  OCC_DOW|OCC_HOUR|OCC_MONTH|OCC_YEAR|PASSENGER|PD_COLLISIONS|PEDESTRIAN|
+----------+-------+--------+---------------+----------+--------------+--------+-----------------+------------------+------------------+----------+--------------------+--------+-------------+---------+--------+---------+--------+---------+-------------+----------+
|       YES|     NO|     D53| GO-20168037252|         0|            NO|     098|               NO|43.685876842418836|-79.39317876160105|        NO|Rosedale-Moore Pa...|  176001|1472616000000|Wednesday|    

In [0]:
# remove following columns: objectID, Hood_158, occ_month, occ_year, occ_dow
toronto_collisions_transformed = toronto_collisions_raw.drop(
    *["OBJECTID", "HOOD_158", "OCC_MONTH", "OCC_YEAR", "OCC_DOW"]
)

In [0]:
# drop duplicates if exists
toronto_collisions_transformed = toronto_collisions_transformed.dropDuplicates(
    subset=["EVENT_UNIQUE_ID"]
)

In [0]:
# drop na/null records if exist in any column
toronto_collisions_transformed = toronto_collisions_transformed.dropna(how="any")

In [0]:
from pyspark.sql.functions import col, greatest
from pyspark.sql.types import StringType

# Get only string columns
string_cols = [
    field.name
    for field in toronto_collisions_transformed.schema.fields
    if isinstance(field.dataType, StringType)
]

# Check rows where any string column has NSA or N/R
nsa_nr_count = toronto_collisions_transformed.filter(
    greatest(
        *[col(c).isin("NSA", "N/R") for c in string_cols]
    )
).count()

total_count = toronto_collisions_transformed.count()

percentage = (nsa_nr_count / total_count) * 100

print("Rows with NSA or N/R:", nsa_nr_count)
print("Total rows:", total_count)
print("Percentage:", percentage)

Rows with NSA or N/R: 34890
Total rows: 255998
Percentage: 13.629012726661927


In [0]:
from pyspark.sql.functions import col, greatest
from pyspark.sql.types import StringType

# Only check string columns, because NSA / N/R are string values
string_cols = [
    field.name
    for field in toronto_collisions_transformed.schema.fields
    if isinstance(field.dataType, StringType)
]

# Rows where any string column has N/R
nr_count = toronto_collisions_transformed.filter(
    greatest(*[col(c).isin("N/R") for c in string_cols])
).count()

# Rows where any string column has NSA
nsa_count = toronto_collisions_transformed.filter(
    greatest(*[col(c).isin("NSA") for c in string_cols])
).count()

# Rows where any string column has either NSA or N/R
nsa_or_nr_count = toronto_collisions_transformed.filter(
    greatest(*[col(c).isin("NSA", "N/R") for c in string_cols])
).count()

total_count = toronto_collisions_transformed.count()

print("Total rows:", total_count)
print("Rows with N/R:", nr_count)
print("Rows with NSA:", nsa_count)
print("Rows with NSA or N/R:", nsa_or_nr_count)
print("Percentage with NSA or N/R:", (nsa_or_nr_count / total_count) * 100)

Total rows: 255998
Rows with N/R: 1409
Rows with NSA: 33847
Rows with NSA or N/R: 34890
Percentage with NSA or N/R: 13.629012726661927


In [0]:
from pyspark.sql.functions import col, greatest
from pyspark.sql.types import StringType

# Only string columns can contain "NSA" or "N/R"
string_cols = [
    field.name
    for field in toronto_collisions_transformed.schema.fields
    if isinstance(field.dataType, StringType)
]

# Remove rows where any string column contains NSA or N/R
bad_rows = toronto_collisions_transformed.filter(
    greatest(*[col(c).isin("NSA", "N/R") for c in string_cols])
)

toronto_collisions_transformed = toronto_collisions_transformed.subtract(bad_rows)

In [0]:
toronto_collisions_transformed = toronto_collisions_transformed.withColumn(
    "DATE_EVENT",
    to_date(from_unixtime(col("OCC_DATE") / 1000))
)


In [0]:
toronto_collisions_transformed = toronto_collisions_transformed.drop("OCC_DATE")

In [0]:
toronto_collisions_transformed = (
    toronto_collisions_transformed.withColumn(
        "FATALITIES", col("FATALITIES").cast("integer")
    )
    .withColumnRenamed("LAT_WGS84", "LATITUDE")
    .withColumnRenamed("LONG_WGS84", "LONGITUDE")
    .withColumnRenamed("NEIGHBOURHOOD_158", "NEIGHBOURHOOD_NAME")
    .withColumn("OCC_HOUR", col("OCC_HOUR").cast("integer"))
    .withColumnRenamed("OCC_HOUR", "HOUR_EVENT")
)

In [0]:
toronto_collisions_transformed.printSchema()

root
 |-- AUTOMOBILE: string (nullable = true)
 |-- BICYCLE: string (nullable = true)
 |-- DIVISION: string (nullable = true)
 |-- EVENT_UNIQUE_ID: string (nullable = true)
 |-- FATALITIES: integer (nullable = true)
 |-- FTR_COLLISIONS: string (nullable = true)
 |-- INJURY_COLLISIONS: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- MOTORCYCLE: string (nullable = true)
 |-- NEIGHBOURHOOD_NAME: string (nullable = true)
 |-- HOUR_EVENT: integer (nullable = true)
 |-- PASSENGER: string (nullable = true)
 |-- PD_COLLISIONS: string (nullable = true)
 |-- PEDESTRIAN: string (nullable = true)
 |-- DATE_EVENT: date (nullable = true)



In [0]:
from pyspark.sql.functions import col

toronto_collisions_transformed = toronto_collisions_transformed.orderBy(
    col("DATE_EVENT").desc()
)

display(toronto_collisions_transformed.limit(20))

AUTOMOBILE,BICYCLE,DIVISION,EVENT_UNIQUE_ID,FATALITIES,FTR_COLLISIONS,INJURY_COLLISIONS,LATITUDE,LONGITUDE,MOTORCYCLE,NEIGHBOURHOOD_NAME,HOUR_EVENT,PASSENGER,PD_COLLISIONS,PEDESTRIAN,DATE_EVENT
YES,NO,D31,GO-20178053126,0,NO,NO,43.75722998520114,-79.51772111197032,NO,Black Creek (24),7,NO,YES,NO,2017-11-02
YES,NO,D31,GO-20178053037,0,YES,NO,43.72243694643958,-79.51346604340414,NO,Oakdale-Beverley Heights (154),10,NO,NO,NO,2017-11-02
YES,NO,D31,GO-20171985304,0,NO,YES,43.717161431240974,-79.53778393242587,NO,Pelmo Park-Humberlea (23),14,YES,NO,NO,2017-11-02
YES,NO,D43,GO-20178053448,0,YES,NO,43.75771699120947,-79.23525552303381,NO,Woburn North (142),8,NO,NO,NO,2017-11-02
YES,NO,D55,GO-20178053111,0,NO,NO,43.65100869136317,-79.34697697013733,NO,South Riverdale (70),5,NO,YES,NO,2017-11-02
YES,NO,D52,GO-20178053128,0,NO,YES,43.64290718201088,-79.38146754290888,NO,St Lawrence-East Bayfront-The Islands (166),7,NO,NO,YES,2017-11-02
YES,NO,D13,GO-20171984028,0,NO,YES,43.70721095752305,-79.4430619697147,NO,Yorkdale-Glen Park (31),8,NO,NO,YES,2017-11-02
YES,NO,D12,GO-20178053199,0,YES,NO,43.694562276535386,-79.48599007384558,NO,Beechborough-Greenbrook (112),10,NO,NO,NO,2017-11-02
YES,NO,D52,GO-20178053455,0,NO,NO,43.65989239899617,-79.39035214362231,NO,Bay-Cloverhill (169),13,NO,YES,NO,2017-11-02
YES,NO,D22,GO-20178053210,0,NO,NO,43.6474101526798,-79.51137889617173,NO,Kingsway South (15),8,NO,YES,NO,2017-11-02


## Save Toronto data to Azure data lake storage

In [0]:
from datetime import datetime

run_date = datetime.today().strftime("%Y%m%d")

toronto_weather_save_path = f"{base_path}/transformed/toronto_weather_transformed_{run_date}"

toronto_weather_transformed.write.format("parquet").mode("overwrite").save(
    toronto_weather_save_path
)

print(f"Saved weather transformed data to: {toronto_weather_save_path}")

Saved weather transformed data to: abfss://traffic-collision@trafficcollision1storage.dfs.core.windows.net/transformed/toronto_weather_transformed_20260517


In [0]:
# Save Toronto traffic collisions data
# Output: traffic-collision/transformed/toronto_traffic_collisions_transformed_YYYYMMDD

toronto_collisions_save_path = f"{base_path}/transformed/toronto_traffic_collisions_transformed_{run_date}"

toronto_collisions_transformed.write.format("parquet").mode("overwrite").save(
    toronto_collisions_save_path
)

print(f"Saved traffic transformed data to: {toronto_collisions_save_path}")

Saved traffic transformed data to: abfss://traffic-collision@trafficcollision1storage.dfs.core.windows.net/transformed/toronto_traffic_collisions_transformed_20260517
